# exp52 round 5 on Colab — SFT 1,200 steps + eval

Port of the Mac Studio (MLX) run to Hugging Face transformers + PEFT. Same data, same LoRA shape, same prompts, same scorer.

**What it does:** answer-only SFT of `google/gemma-4-E4B-it` on 4,000 synthetic QC cases (no real cases in training), then scores the adapter on synthetic validation, the 197 real dev cases, and the 1,200 real test cases with chance-corrected metrics.

**Runtime:** A100 (Runtime → Change runtime type). Rough estimates: training ~45–60 min, evals ~10–15 min. A T4 (16 GB) can't hold the bf16 model for training.

**Mac Studio reference (dev, n=197):** base composite skill 0.019 · SFT-300 0.132 · SFT-600 **0.200** (T1 κ 0.424, T3 κ 0.219). Test gate of SFT-300→GRPO-80: composite 0.188 vs base 0.011.

**Before running:** upload the contents of this kit to `MyDrive/Projects/snap-qc-finetune/`, and accept the Gemma license on Hugging Face for `google/gemma-4-E4B-it` (you'll need an HF token).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/Projects/snap-qc-finetune'   # change if you put the folder elsewhere
import os, sys
os.chdir(ROOT)
sys.path[:0] = [f'{ROOT}/eval', f'{ROOT}/src']
os.environ['PYTHONPATH'] = f'{ROOT}/src:{ROOT}/eval'
!ls -la {ROOT} {ROOT}/data

In [ ]:
!pip -q install -U transformers peft accelerate
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))

In [ ]:
# HF_TOKEN: Colab secret (key icon in the left sidebar, notebook access on) or an environment variable
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not found: add it as a Colab secret named HF_TOKEN'
os.environ['HF_TOKEN'] = HF_TOKEN   # transformers / huggingface_hub read this too
login(token=HF_TOKEN)

## Config — matches the Mac Studio runs

MLX LoRA `scale` multiplies the adapter output directly; PEFT scales by `lora_alpha / r`, so scale 20 at rank 16 is `lora_alpha = 320`.
The MLX adapter trained the last 16 of 42 decoder layers (26–41): `q_proj`, `o_proj`, the three MLP projections, and Gemma's per-layer input gate/projection — 112 LoRA modules, ~13.8M trainable parameters.

In [ ]:
CFG = dict(
    model_id='google/gemma-4-E4B-it',
    steps=1200, lr=1e-5, batch=4, max_len=4096, seed=52,
    r=16, alpha=320, dropout=0.05,
    eval_every=100, val_examples=40, save_every=200,
    run_name='sft_lr1e-5_1200_colab',
)
TARGET_REGEX = (r'.*language_model.*layers\.(2[6-9]|3[0-9]|4[01])\.'
                r'(self_attn\.(q_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj)|per_layer_input_gate|per_layer_projection)$')
OUT = f"{ROOT}/results/{CFG['run_name']}"
os.makedirs(OUT, exist_ok=True)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(CFG['model_id'])
try:
    model = AutoModelForCausalLM.from_pretrained(CFG['model_id'], torch_dtype=torch.bfloat16, device_map='cuda')
except Exception as e:   # some Gemma 4 releases only register the multimodal class
    print('AutoModelForCausalLM failed, trying AutoModelForImageTextToText:', e)
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(CFG['model_id'], torch_dtype=torch.bfloat16, device_map='cuda')
model.gradient_checkpointing_enable()
model.config.use_cache = False

from peft import LoraConfig, get_peft_model
model = get_peft_model(model, LoraConfig(r=CFG['r'], lora_alpha=CFG['alpha'], lora_dropout=CFG['dropout'],
                                         target_modules=TARGET_REGEX, task_type='CAUSAL_LM'))
n_mod = sum(1 for n, _ in model.named_modules() if n.endswith('lora_A'))
model.print_trainable_parameters()
print('LoRA modules:', n_mod, '(MLX run had 112)')
if n_mod != 112:
    print('WARNING: module names differ from the MLX model; check TARGET_REGEX against model.named_modules()')

## Data — prompts rendered exactly as the eval serves them

Thinking off, single user turn. The training target is the gold answer block plus Gemma's end-of-turn marker (`<turn|>\n`); the prompt is masked out of the loss.

In [ ]:
import json, random
def render_prompt(p):
    return tok.apply_chat_template([{'role': 'user', 'content': p}], add_generation_prompt=True,
                                   tokenize=False, enable_thinking=False)
# sanity: prompt is a prefix of the full conversation, completion ends with the turn marker
_f = tok.apply_chat_template([{'role': 'user', 'content': 'X'}, {'role': 'assistant', 'content': 'Y'}],
                             tokenize=False, enable_thinking=False)
END = _f[len(render_prompt('X')) + 1:]
assert _f.startswith(render_prompt('X')) and END.startswith('<turn|>'), (_f, END)

def encode(rec):
    p = tok(render_prompt(rec['prompt']), add_special_tokens=False)['input_ids']
    c = tok(rec['completion'] + END, add_special_tokens=False)['input_ids']
    ids = (p + c)[:CFG['max_len']]
    return {'input_ids': ids, 'labels': ([-100] * len(p) + c)[:CFG['max_len']]}

train = [encode(json.loads(l)) for l in open(f'{ROOT}/data/sft_train.jsonl')]
valid = [encode(json.loads(l)) for l in open(f'{ROOT}/data/sft_valid.jsonl')][:CFG['val_examples']]
print(len(train), len(valid), 'max len', max(len(x['input_ids']) for x in train))

def collate(batch):
    m = max(len(x['input_ids']) for x in batch)
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    ids = [x['input_ids'] + [pad] * (m - len(x['input_ids'])) for x in batch]
    lab = [x['labels'] + [-100] * (m - len(x['labels'])) for x in batch]
    att = [[1] * len(x['input_ids']) + [0] * (m - len(x['input_ids'])) for x in batch]
    return {'input_ids': torch.tensor(ids), 'labels': torch.tensor(lab), 'attention_mask': torch.tensor(att)}

## Train — 1,200 steps, constant lr 1e-5, batch 4 (MLX used Adam, no schedule, no weight decay)

In [ ]:
from transformers import Trainer, TrainingArguments
args = TrainingArguments(
    output_dir=OUT, max_steps=CFG['steps'], per_device_train_batch_size=CFG['batch'],
    per_device_eval_batch_size=4, learning_rate=CFG['lr'], lr_scheduler_type='constant',
    warmup_steps=0, weight_decay=0.0, optim='adamw_torch', bf16=True,
    logging_steps=10, eval_strategy='steps', eval_steps=CFG['eval_every'],
    save_steps=CFG['save_every'], save_total_limit=6, seed=CFG['seed'],
    report_to='none', remove_unused_columns=False, gradient_checkpointing=True,
)
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=valid, data_collator=collate)
trainer.train()            # resumes with trainer.train(resume_from_checkpoint=True) after a disconnect
model.save_pretrained(f'{OUT}/adapter_final')

## Eval helpers — greedy, max 64 new tokens, stop at end-of-turn (same as the MLX quick eval)

In [ ]:
import run_lm, labels as L, subprocess
model.eval(); model.config.use_cache = True
tok.padding_side = 'left'
STOP = [tok.eos_token_id, tok.convert_tokens_to_ids('<turn|>')]

@torch.no_grad()
def generate(prompts, bs=16):
    outs = []
    for i in range(0, len(prompts), bs):
        enc = tok([render_prompt(p) for p in prompts[i:i+bs]], return_tensors='pt',
                  padding=True, add_special_tokens=False).to('cuda')
        g = model.generate(**enc, max_new_tokens=64, do_sample=False, eos_token_id=STOP)
        outs += tok.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        print(f'{min(i+bs, len(prompts))}/{len(prompts)}', end='\r')
    return outs

items = json.load(open(f'{ROOT}/eval/items.json'))
dev = [d for d in items if d['split'] == 'dev']
seen, EX, ex_ids = set(), [], set()      # few-shot examples: first correct/over/under in dev order (run_lm rule)
for d in dev:
    if d['finding'] not in seen:
        seen.add(d['finding']); EX.append((d['fields'], d['labels'])); ex_ids.add(d['item_id'])
    if len(EX) == 3: break

def eval_split(split_items, out_path):
    raws = generate([run_lm.build_class_prompt(it['fields'], 'jargon', EX) for it in split_items])
    with open(out_path, 'w') as fh:
        for it, raw in zip(split_items, raws):
            parsed, ok = run_lm.parse_class(raw)
            fh.write(json.dumps({'item_id': it['item_id'], 'kind': 'class', 'raw': raw,
                                 'parsed': parsed, 'parse_ok': ok}) + '\n')
    return out_path

## 1) Synthetic validation — can the model learn the task in-distribution?

In [ ]:
import score as S
syn = [json.loads(l) for l in open(f'{ROOT}/data/synthetic_valid.jsonl')][:120]
preds = [run_lm.parse_class(r)[0] for r in generate([r['prompt'] for r in syn])]
gold = [json.loads(r['answer']) for r in syn]
ex = []
for t in ('t1', 't2', 't3', 't4', 't5'):
    pairs = [(p.get(t), g[t]) for p, g in zip(preds, gold) if g[t] is not None]
    cc = S.chance_corrected(pairs, L.TASKS[t]['values'])
    ex.append(cc['excess'])
    print(f"syn_{t} macroF1 {S.macro_f1(pairs, L.TASKS[t]['values']):.3f} kappa {cc['kappa']:.3f} excess {cc['excess']:.3f} p {cc['p_value']:.3f}")
print('synthetic mean excess:', round(sum(ex) / 5, 3), '(MLX: base 0.062, SFT-600 0.118)')

## 2) Real dev (197 cases) — the iteration metric

In [ ]:
dev_items = [d for d in dev if d['item_id'] not in ex_ids]
dev_out = eval_split(dev_items, f'{OUT}/dev_preds.jsonl')
open(f'{OUT}/dev.ids', 'w').write('\n'.join(d['item_id'] for d in dev_items) + '\n')
!python {ROOT}/eval/score.py --items {ROOT}/eval/items.json --preds {dev_out} --split dev --ids {OUT}/dev.ids | tail -8

## 3) Real test (1,200 cases) — the gate. Run once, on the configuration you keep.

`--base-preds` here is the MLX bf16 base from the Mac Studio. For a strictly matched comparison (same software stack), set `RUN_BASE = True` to regenerate base predictions on Colab with the adapter disabled.

In [ ]:
test_items = [d for d in items if d['split'] in ('test_main', 'test_2024')]
test_out = eval_split(test_items, f'{OUT}/test_preds.jsonl')

RUN_BASE = False
base = f'{ROOT}/eval/preds_base_e4b_mlx_bf16_test.jsonl'
if RUN_BASE:
    with model.disable_adapter():
        base = eval_split(test_items, f'{ROOT}/results/base_e4b_hf_bf16_test.jsonl')
for split in ('test', 'test_2024'):
    !python {ROOT}/eval/score.py --items {ROOT}/eval/items.json --preds {test_out} --base-preds {base} --ref-preds {ROOT}/eval/preds_base_31b_test.jsonl --split {split} --out {OUT}/gate_{split}.json | tail -10